<a href="https://colab.research.google.com/github/kjm5416-ship-it/-secom-anomaly-detection/blob/main/02_%EC%A0%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/sharmaroshan/SECOM-Detecting-Defected-Items/master/uci-secom.csv"
df = pd.read_csv(url)

X = df.drop(columns=['Time', 'Pass/Fail'])
y = (df['Pass/Fail'] == 1).astype(int)

print("X (센서값):", X.shape)
print("y (정답) 불량 개수:", y.sum())

X (센서값): (1567, 590)
y (정답) 불량 개수: 104


In [ ]:
# 1단계: 값이 항상 똑같은 센서 제거
const_cols = X.columns[X.nunique() <= 1]
X = X.drop(columns=const_cols)
print("제거 — 항상 같은 값:", len(const_cols), "개")

# 2단계: 절반 이상 비어 있는 센서 제거
missing = X.isna().mean()
high_missing = X.columns[missing > 0.5]
X = X.drop(columns=high_missing)
print("제거 — 절반 이상 결측:", len(high_missing), "개")

print("남은 센서:", X.shape[1], "개")

제거 — 항상 같은 값: 116 개
제거 — 절반 이상 결측: 28 개
남은 센서: 446 개


In [ ]:
print("아직 빈 값이 있는 센서:", (X.isna().sum() > 0).sum(), "개")
print("빈 칸 총 개수:", X.isna().sum().sum())

아직 빈 값이 있는 센서: 394 개
빈 칸 총 개수: 10868


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("학습용:", X_train.shape, "| 불량", y_train.sum(), "개")
print("시험용:", X_test.shape, "| 불량", y_test.sum(), "개")
print("불량률 — 학습 %.2f%% / 시험 %.2f%%" % (y_train.mean()*100, y_test.mean()*100))

학습용: (1253, 446) | 불량 83 개
시험용: (314, 446) | 불량 21 개
불량률 — 학습 6.62% / 시험 6.69%


In [ ]:
from sklearn.impute import SimpleImputer
import pandas as pd

imputer = SimpleImputer(strategy='median')

X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
X_test  = pd.DataFrame(imputer.transform(X_test),      columns=X.columns)

print("남은 빈 칸 — 학습:", X_train.isna().sum().sum(), "/ 시험:", X_test.isna().sum().sum())

남은 빈 칸 — 학습: 0 / 시험: 0


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)

print("표준화 후 — 평균 %.3f / 표준편차 %.3f" % (X_train.values.mean(), X_train.values.std()))
print("최종 학습용:", X_train.shape)
print("최종 시험용:", X_test.shape)

표준화 후 — 평균 0.000 / 표준편차 1.000
최종 학습용: (1253, 446)
최종 시험용: (314, 446)


In [ ]:
print("X_train 번호:", list(X_train.index[:5]))
print("y_train 번호:", list(y_train.index[:5]))

X_train 번호: [0, 1, 2, 3, 4]
y_train 번호: [1198, 436, 635, 996, 782]


In [ ]:
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print("맞춘 후 y_train 번호:", list(y_train.index[:5]))
print("길이 확인 —", len(X_train), len(y_train), "/", len(X_test), len(y_test))

맞춘 후 y_train 번호: [0, 1, 2, 3, 4]
길이 확인 — 1253 1253 / 314 314


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

path = '/content/drive/MyDrive/도전학기'
os.makedirs(path, exist_ok=True)

# 원본 백업 — GitHub 링크가 죽어도 이건 남는다
df.to_csv(f'{path}/secom_원본.csv', index=False)

# 전처리 결과
X_train.to_csv(f'{path}/X_train.csv', index=False)
X_test.to_csv(f'{path}/X_test.csv',  index=False)
y_train.to_csv(f'{path}/y_train.csv', index=False)
y_test.to_csv(f'{path}/y_test.csv',  index=False)

print("저장 완료")
print(os.listdir(path))

저장 완료
['secom_원본.csv', 'X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv']
